# preprocess_temporal_v3.ipynb
**Runs from:** `backend/data generation/`

## Preprocessing philosophy — group-based scaling

| Group | Features | Treatment |
|-------|----------|-----------|
| **Targets** | travel_time_ratio, congestion_level | Separate StandardScaler each |
| **Continuous temporal** | hourly_rainfall_mm, incidents_nearby, rain_log1p, incident_load | StandardScaler (scaler_cont) |
| **Binary flags** | road_closure, incident, monsoon, disruption, holidays, is_weekend, is_peak | **No scaling** (already 0/1) |
| **One-hot incident type** | 6 dummies for Accident/Rain/Jam/Closed/Works/Flood | **No scaling** |
| **Ordinal** | incident_severity (0-4) | **No scaling** |
| **Cyclic** | time_of_day sin/cos, day_of_week sin/cos | **No scaling** (already [-1,1]) |
| **Static continuous** | road_length, susceptibility, signals_per_km, etc. | Separate StandardScaler (scaler_static) |
| **Static binary** | oneway | **No scaling** |

**Output:** 30 separate batch files (~55 MB each) + corridor_batch_map.pkl
**Never** creates a single combined parquet. Peak RAM ~500 MB.

In [1]:
import os, glob, pickle, time, warnings, gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path
warnings.filterwarnings('ignore')

RAW_TS_DIR   = 'data/timeseries'
STATIC_PATH  = 'data/static/edges_static.parquet'
OUT_DIR      = 'data/processed'
SCALER_DIR   = 'data/scalers'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

DROP_COLS = ['speed_ratio', 'current_speed', 'current_travel_time',
             'delay_seconds', 'confidence']
TRAIN_END = 107; VAL_END = 143; TEST_END = 167; TTR_CLIP = 5.0

TARGET_COLS    = ['travel_time_ratio', 'congestion_level']
CONT_TEMPORAL  = ['hourly_rainfall_mm', 'incidents_nearby', 'rain_log1p', 'incident_load']
BINARY_TEMPORAL= ['road_closure','incident','monsoon_active','local_train_disruption',
                  'is_public_holiday','school_holiday','is_weekend','is_peak_hour']
ONEHOT_INC     = ['inc_type_accident','inc_type_rain','inc_type_jam',
                  'inc_type_road_closed','inc_type_road_works','inc_type_flooding']
ORDINAL_COLS   = ['incident_severity']
CYCLIC_COLS    = ['time_of_day_sin','time_of_day_cos','day_of_week_sin','day_of_week_cos']
STATIC_SCALED  = ['road_type_enc','num_lanes','road_length','susceptibility',
                  'traffic_signal_count','intersection_count','signals_per_km']
STATIC_BINARY  = ['oneway']

ALL_FEAT_COLS = (TARGET_COLS + CONT_TEMPORAL + BINARY_TEMPORAL + ONEHOT_INC +
                 ORDINAL_COLS + CYCLIC_COLS + STATIC_SCALED + STATIC_BINARY)
RAW_FEAT_DIM = len(ALL_FEAT_COLS)
print(f'Features: {RAW_FEAT_DIM}, IN_DIM={RAW_FEAT_DIM+32}')
for i,c in enumerate(ALL_FEAT_COLS): print(f'  [{i:2d}] {c}')

Features: 33, IN_DIM=65
  [ 0] travel_time_ratio
  [ 1] congestion_level
  [ 2] hourly_rainfall_mm
  [ 3] incidents_nearby
  [ 4] rain_log1p
  [ 5] incident_load
  [ 6] road_closure
  [ 7] incident
  [ 8] monsoon_active
  [ 9] local_train_disruption
  [10] is_public_holiday
  [11] school_holiday
  [12] is_weekend
  [13] is_peak_hour
  [14] inc_type_accident
  [15] inc_type_rain
  [16] inc_type_jam
  [17] inc_type_road_closed
  [18] inc_type_road_works
  [19] inc_type_flooding
  [20] incident_severity
  [21] time_of_day_sin
  [22] time_of_day_cos
  [23] day_of_week_sin
  [24] day_of_week_cos
  [25] road_type_enc
  [26] num_lanes
  [27] road_length
  [28] susceptibility
  [29] traffic_signal_count
  [30] intersection_count
  [31] signals_per_km
  [32] oneway


In [2]:
# ── Load static ───────────────────────────────────────────────────────────────
df_static = pd.read_parquet(STATIC_PATH)
df_static['free_flow_travel_time'] = (
    df_static['road_length'] / (df_static['free_flow_speed'] * 1000/3600)
).clip(lower=0.5).astype(np.float32)
if 'signals_per_km' not in df_static.columns:
    df_static['signals_per_km'] = (df_static['traffic_signal_count'] /
        (df_static['road_length']/1000).clip(lower=0.01)).astype(np.float32)
df_static['oneway'] = df_static['oneway'].astype(np.float32)
df_static.to_parquet(f'{OUT_DIR}/edges_static_scaled.parquet', index=False)

ALL_STATIC = STATIC_SCALED + STATIC_BINARY
static_lkp = df_static.set_index('edge_id')[ALL_STATIC + ['corridor_id']].to_dict('index')
timestamps = pd.date_range(start='2024-07-01', periods=168, freq='h')
ts_to_hour = {ts: i for i, ts in enumerate(timestamps)}
batch_files = sorted(glob.glob(f'{RAW_TS_DIR}/batch_*.parquet'))
inc_map = {1:'inc_type_accident',4:'inc_type_rain',6:'inc_type_jam',
           8:'inc_type_road_closed',9:'inc_type_road_works',11:'inc_type_flooding'}
print(f'Static: {len(df_static):,} edges, Batches: {len(batch_files)}')

Static: 145,265 edges, Batches: 30


In [3]:
# ── Batch processor ───────────────────────────────────────────────────────────
def process_batch(bf):
    df = pd.read_parquet(bf)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
    df = df.drop(columns=['free_flow_speed','free_flow_travel_time'], errors='ignore')
    df['travel_time_ratio'] = df['travel_time_ratio'].clip(upper=TTR_CLIP)
    df['hour_index'] = df['timestamp'].map(ts_to_hour).astype(np.int16)
    df['rain_log1p'] = np.log1p(df['hourly_rainfall_mm']).astype(np.float32)
    df['incident_load'] = (df['incident'] * df['incident_severity']).astype(np.float32)
    df['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(np.int8)
    h = df['timestamp'].dt.hour
    df['is_peak_hour'] = (((h>=7)&(h<=10))|((h>=17)&(h<=20))).astype(np.int8)
    for code, col in inc_map.items():
        df[col] = (df['incident_type'] == code).astype(np.int8)
    df = df.drop(columns=['incident_type'])
    for col in ALL_STATIC + ['corridor_id']:
        dt = np.int16 if col == 'corridor_id' else np.float32
        df[col] = df['edge_id'].map(lambda e, c=col: static_lkp.get(e,{}).get(c,0)).astype(dt)
    return df[['edge_id','hour_index','corridor_id'] + ALL_FEAT_COLS]

test = process_batch(batch_files[0])
assert list(test.columns[3:]) == ALL_FEAT_COLS, 'Column mismatch!'
print(f'Batch 0: {test.shape}, {test.memory_usage(deep=True).sum()/1e6:.0f} MB')
del test; gc.collect()

Batch 0: (840000, 36), 100 MB


0

In [4]:
# ── PASS 1: Fit scalers on train data ─────────────────────────────────────────
print('Pass 1: Fitting scalers...')
t0 = time.perf_counter()
acc = {'ttr':[], 'cong':[], 'cont':[], 'stat':[]}
for i, bf in enumerate(batch_files):
    df = process_batch(bf)
    tr = df[df['hour_index'] <= TRAIN_END]
    if len(tr):
        acc['ttr'].append(tr[['travel_time_ratio']].values.astype(np.float32))
        acc['cong'].append(tr[['congestion_level']].values.astype(np.float32))
        acc['cont'].append(tr[CONT_TEMPORAL].values.astype(np.float32))
        acc['stat'].append(tr[STATIC_SCALED].values.astype(np.float32))
    del df, tr; gc.collect()
    if (i+1)%10==0: print(f'  {i+1}/{len(batch_files)}')

scaler_ttr = StandardScaler().fit(np.concatenate(acc['ttr']))
scaler_cong = StandardScaler().fit(np.concatenate(acc['cong']))
scaler_cont = StandardScaler().fit(np.concatenate(acc['cont']))
scaler_static = StandardScaler().fit(np.concatenate(acc['stat']))
del acc; gc.collect()

for n,s in [('scaler_ttr',scaler_ttr),('scaler_cong',scaler_cong),
            ('scaler_cont',scaler_cont),('scaler_static',scaler_static)]:
    with open(f'{SCALER_DIR}/{n}.pkl','wb') as f: pickle.dump(s,f)

feat_meta = {'ALL_FEAT_COLS':ALL_FEAT_COLS,'RAW_FEAT_DIM':RAW_FEAT_DIM,
    'TARGET_COLS':TARGET_COLS,'CONT_TEMPORAL':CONT_TEMPORAL,'BINARY_TEMPORAL':BINARY_TEMPORAL,
    'ONEHOT_INC':ONEHOT_INC,'ORDINAL_COLS':ORDINAL_COLS,'CYCLIC_COLS':CYCLIC_COLS,
    'STATIC_SCALED':STATIC_SCALED,'STATIC_BINARY':STATIC_BINARY,
    'TARGET_IDX_TTR':0,'TARGET_IDX_CONG':1}
with open(f'{SCALER_DIR}/feat_meta.pkl','wb') as f: pickle.dump(feat_meta,f)

print(f'Scalers fit in {time.perf_counter()-t0:.1f}s')
print(f'  TTR:  mean={scaler_ttr.mean_[0]:.4f}, std={scaler_ttr.scale_[0]:.4f}')
print(f'  Cong: mean={scaler_cong.mean_[0]:.4f}, std={scaler_cong.scale_[0]:.4f}')

Pass 1: Fitting scalers...
  10/30
  20/30
  30/30
Scalers fit in 98.8s
  TTR:  mean=1.3351, std=0.3898
  Cong: mean=0.2141, std=0.1439


In [5]:
# ── PASS 2: Transform → save 30 SEPARATE batch files ─────────────────────────
# Also build corridor_batch_map: {corridor_id → set of batch indices}
print('\nPass 2: Transforming and saving separate batch files...')
t0 = time.perf_counter()
corridor_batch_map = {}  # cid → set of batch indices

for i, bf in enumerate(batch_files):
    df = process_batch(bf)

    # Scale
    df['travel_time_ratio'] = scaler_ttr.transform(
        df[['travel_time_ratio']].values.astype(np.float32)).astype(np.float32)
    df['congestion_level'] = scaler_cong.transform(
        df[['congestion_level']].values.astype(np.float32)).astype(np.float32)
    df[CONT_TEMPORAL] = scaler_cont.transform(
        df[CONT_TEMPORAL].values.astype(np.float32)).astype(np.float32)
    df[STATIC_SCALED] = scaler_static.transform(
        df[STATIC_SCALED].values.astype(np.float32)).astype(np.float32)

    # Verify unscaled features
    for col in BINARY_TEMPORAL + ONEHOT_INC:
        assert set(df[col].unique()).issubset({0,1,0.0,1.0}), f'Batch {i}: {col} not binary'
    for col in CYCLIC_COLS:
        assert df[col].min() >= -1.01 and df[col].max() <= 1.01, f'Batch {i}: {col} OOB'

    # Track which corridors are in this batch
    for cid in df['corridor_id'].unique():
        corridor_batch_map.setdefault(int(cid), set()).add(i)

    # Save as SEPARATE file (never combined)
    out = f'{OUT_DIR}/batch_{i:04d}.parquet'
    df.to_parquet(out, index=False, compression='snappy')
    del df; gc.collect()
    if (i+1)%10==0: print(f'  {i+1}/{len(batch_files)}')

# Convert sets to sorted lists for pickle
corridor_batch_map = {k: sorted(v) for k, v in corridor_batch_map.items()}
with open(f'{OUT_DIR}/corridor_batch_map.pkl', 'wb') as f:
    pickle.dump(corridor_batch_map, f)

elapsed = time.perf_counter() - t0
print(f'\nSaved {len(batch_files)} separate batch files → {OUT_DIR}/batch_XXXX.parquet')
print(f'Saved corridor_batch_map.pkl ({len(corridor_batch_map)} corridors)')
print(f'Time: {elapsed:.1f}s')

# Split statistics (scan one batch only to avoid loading all)
sample = pd.read_parquet(f'{OUT_DIR}/batch_0000.parquet')
n_edges_b0 = sample['edge_id'].nunique()
for name, lo, hi in [('Train',0,TRAIN_END),('Val',TRAIN_END+1,VAL_END),('Test',VAL_END+1,TEST_END)]:
    n = ((sample['hour_index']>=lo)&(sample['hour_index']<=hi)).sum()
    print(f'{name}: hours {lo}-{hi} → {n:,} rows/batch × {len(batch_files)} batches')
del sample; gc.collect()


Pass 2: Transforming and saving separate batch files...
  10/30
  20/30
  30/30

Saved 30 separate batch files → data/processed/batch_XXXX.parquet
Saved corridor_batch_map.pkl (24 corridors)
Time: 119.5s
Train: hours 0-107 → 540,000 rows/batch × 30 batches
Val: hours 108-143 → 180,000 rows/batch × 30 batches
Test: hours 144-167 → 120,000 rows/batch × 30 batches


0

In [6]:
# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'  TEMPORAL PREPROCESSING COMPLETE')
print(f'{"="*70}')
print(f'  Output:  {OUT_DIR}/batch_0000.parquet ... batch_{len(batch_files)-1:04d}.parquet')
print(f'  Static:  {OUT_DIR}/edges_static_scaled.parquet')
print(f'  Map:     {OUT_DIR}/corridor_batch_map.pkl')
print(f'  Scalers: {SCALER_DIR}/')
print(f'\n  Feature layout ({RAW_FEAT_DIM} features):')
i=0
for grp,cols,method in [('Targets',TARGET_COLS,'scaled separately'),
    ('Continuous',CONT_TEMPORAL,'StandardScaler'),('Binary',BINARY_TEMPORAL,'NO scaling'),
    ('One-hot',ONEHOT_INC,'NO scaling'),('Ordinal',ORDINAL_COLS,'NO scaling'),
    ('Cyclic',CYCLIC_COLS,'NO scaling'),('Static cont',STATIC_SCALED,'StandardScaler'),
    ('Static bin',STATIC_BINARY,'NO scaling')]:
    print(f'    [{i}:{i+len(cols)}] {grp:15s} → {method}')
    i += len(cols)


  TEMPORAL PREPROCESSING COMPLETE
  Output:  data/processed/batch_0000.parquet ... batch_0029.parquet
  Static:  data/processed/edges_static_scaled.parquet
  Map:     data/processed/corridor_batch_map.pkl
  Scalers: data/scalers/

  Feature layout (33 features):
    [0:2] Targets         → scaled separately
    [2:6] Continuous      → StandardScaler
    [6:14] Binary          → NO scaling
    [14:20] One-hot         → NO scaling
    [20:21] Ordinal         → NO scaling
    [21:25] Cyclic          → NO scaling
    [25:32] Static cont     → StandardScaler
    [32:33] Static bin      → NO scaling
